#### 1. 라이브러리 임포트

In [1]:
import random
import numpy as np
from pathlib import Path
import pandas as pd
import tensorflow as tf
from transformers import RobertaTokenizerFast, TFRobertaModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

I0000 00:00:1780543311.930000   58052 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780543311.958871   58052 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780543312.741165   58052 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


#### 3. 데이터 로드

In [2]:
# 데이터 경로
dataset_name = "final_data"
parquet_path = Path(f"{dataset_name}.parquet")

# Parquet 파일 읽기
df = pd.read_parquet(parquet_path)

# 데이터 크기 확인
print(f"데이터 크기: {df.shape}")
df.sample(5)

데이터 크기: (70000, 7)


,review_text,fake,basic_linguistic_list,readability_list,sentiment_list,behavioral_list,clean_text
43494,Midtown. Let's face it: you're going to be pay...,1,"[203.0, 127.0, 10.0, 800.0, 641.0, 19.0, 76.0,...","[11.20814326018867, 57.33219767441861, 8.46735...","[0.1961538461538462, 0.6192307692307693, 1.0, ...","[2.0, 0.0, 0.0, 0.0, 0.6931471805599453, 0.5, ...",midtown. let us face it you are going to be pa...
62001,"Love their food! Filling, yummy, readily avail...",0,"[19.0, 9.0, 2.0, 66.0, 54.0, 3.0, 4.0, 1.0, 3....","[0.0, 23.66750000000002, 11.07611111111111, 15...","[0.5625, 0.5, 0.0, 0.0, 2.0, 0.0, 0.0]","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0]",love their food filling yummy readily availabl...
48956,Great food. Â Horrible service. Â Food is fant...,1,"[14.0, 9.0, 3.0, 52.0, 41.0, 2.0, 6.0, 2.0, 2....","[0.0, 70.66750000000002, 4.520555555555557, 10...","[0.0666666666666667, 0.8833333333333333, 0.0, ...","[5.0, 0.0, 0.0, 0.0, 1.0549201679861442, 0.64,...",great food. horrible service. food is fantastic.
61052,The place is absolutely elegant. Â The menu is...,1,"[56.0, 38.0, 5.0, 201.0, 161.0, 5.0, 28.0, 8.0...","[9.516144504307135, 69.40300675675678, 5.87695...","[0.45, 0.7892857142857144, 0.0, 0.0, 3.0, 1.0,...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0]",the place is absolutely elegant. the menu is r...
61670,MALAGUETA [it ought to be in capital letters] ...,1,"[115.0, 77.0, 4.0, 413.0, 331.0, 12.0, 53.0, 1...","[13.02386679866686, 60.94560064935068, 9.54087...","[0.5941558441558441, 0.6614718614718614, 0.0, ...","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 0.0]",malagueta it ought to be in capital letters is...


#### 4. Train / Validation / Test 분할

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["fake"]), df["fake"].astype("float32"), test_size=0.2, stratify=df["fake"], random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.125, stratify=y_train, random_state=42
)

#### 6. Tokenize

In [4]:
MAX_LEN = 256

tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

train_enc = tokenizer(list(X_train["clean_text"]), padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors="np")
val_enc = tokenizer(list(X_val["clean_text"]), padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors="np")
test_enc = tokenizer(list(X_test["clean_text"]), padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors="np")

train_inputs = {"input_ids": train_enc["input_ids"], "attention_mask": train_enc["attention_mask"]}
val_inputs = {"input_ids": val_enc["input_ids"], "attention_mask": val_enc["attention_mask"]}
test_inputs = {"input_ids": test_enc["input_ids"], "attention_mask": test_enc["attention_mask"]}

#### 7. RoBerta 함수

In [5]:
roberta = TFRobertaModel.from_pretrained('roberta-base')
roberta.trainable = False

ids = tf.keras.Input((MAX_LEN,), dtype=tf.int32, name='input_ids')
mask = tf.keras.Input((MAX_LEN,), dtype=tf.int32, name='attention_mask')

x = roberta(ids, attention_mask=mask).last_hidden_state
x = tf.keras.layers.Flatten()(x)

x = tf.keras.layers.Dense(2048, activation='gelu')(x)
x = tf.keras.layers.Dense(1024, activation='gelu')(x)
x = tf.keras.layers.Dense(512, activation='gelu')(x)
x = tf.keras.layers.Dense(256, activation='gelu')(x)
x = tf.keras.layers.Dense(128, activation='gelu')(x)
out = tf.keras.layers.Dense(1, activation='sigmoid')(x)

I0000 00:00:1780543323.086270   58052 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22149 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFRobertaModel: ['lm_head.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight', 'roberta.embeddings.position_ids', 'lm_head.layer_norm.bias']
- This IS expected if you are initializing TFRobertaModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFRobertaModel 

#### 8. 모델 생성

In [6]:
model = tf.keras.Model(inputs={'input_ids': ids, 'attention_mask': mask}, outputs=out)

model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_ids (InputLayer)      [(None, 256)]                0         []                            
                                                                                                  
 attention_mask (InputLayer  [(None, 256)]                0         []                            
 )                                                                                                
                                                                                                  
 tf_roberta_model (TFRobert  TFBaseModelOutputWithPooli   1246456   ['input_ids[0][0]',           
 aModel)                     ngAndCrossAttentions(last_   32         'attention_mask[0][0]']      
                             hidden_state=(None, 256, 7                                       

#### 9. 모델 컴파일

In [7]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss="binary_crossentropy", metrics=["accuracy"])

#### 10. 콜백 설정

In [8]:
callbacks = [tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]

#### 11. 모델 학습

In [9]:
history = model.fit(x={"input_ids": train_enc["input_ids"], "attention_mask": train_enc["attention_mask"]}, y=y_train, 
                    validation_data=({"input_ids": val_enc["input_ids"], "attention_mask": val_enc["attention_mask"]}, y_val), epochs=20, batch_size=32, callbacks=callbacks, verbose=1)

Epoch 1/20


I0000 00:00:1780543343.011682   58236 service.cc:153] XLA service 0x71c4c831e870 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1780543343.011697   58236 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4090, Compute Capability 8.9 (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.23.0)
I0000 00:00:1780543343.015562   58236 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1780543343.028516   58236 cuda_dnn.cc:461] Loaded cuDNN version 92300
I0000 00:00:1780543343.064847   58236 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1532/1532 [==============================] - 168s 106ms/step - loss: 0.6393 - accuracy: 0.6348 - val_loss: 0.6146 - val_accuracy: 0.6597
Epoch 2/20
1532/1532 [==============================] - 160s 105ms/step - loss: 0.5952 - accuracy: 0.6760 - val_loss: 0.6198 - val_accuracy: 0.6629
Epoch 3/20
1532/1532 [==============================] - 160s 105ms/step - loss: 0.5445 - accuracy: 0.7103 - val_loss: 0.6871 - val_accuracy: 0.6407
Epoch 4/20
1532/1532 [==============================] - 160s 105ms/step - loss: 0.4913 - accuracy: 0.7426 - val_loss: 0.6969 - val_accuracy: 0.6416
Epoch 5/20
1532/1532 [==============================] - 160s 105ms/step - loss: 0.4588 - accuracy: 0.7600 - val_loss: 0.7845 - val_accuracy: 0.6343
Epoch 6/20
1532/1532 [==============================] - 160s 105ms/step - loss: 0.4276 - accuracy: 0.7775 - val_loss: 0.8553 - val_accuracy: 0.6521


#### 12. 예측 및 성능 계산

In [10]:
y_pred_prob = model.predict({"input_ids": test_enc["input_ids"], "attention_mask": test_enc["attention_mask"]}, batch_size=32)
y_pred = (y_pred_prob > 0.5).astype(int)

print(f"acc  : {accuracy_score(y_test, y_pred):.4f}")
print(f"prec : {precision_score(y_test, y_pred):.4f}")
print(f"rec  : {recall_score(y_test, y_pred):.4f}")
print(f"f1   : {f1_score(y_test, y_pred):.4f}")

438/438 [==============================] - 31s 69ms/step
acc  : 0.6570
prec : 0.6746
rec  : 0.6067
f1   : 0.6388
